# Workshop introductie pydov

## Wat is pydov?

- Een Python package om gemakkelijk DOV data te kunnen gebruiken in andere scripts en tools
  - Zoeken op attributen en locatie​
  - Combineren met andere datasets
  - Resultaten beschikbaar in een Pandas DataFrame​

- Een referentie client implementatie van onze metadata, WFS en XML services

- Een community project
  - Open ontwikkeling op GitHub:​ https://github.com/DOV-Vlaanderen/pydov/
  - Open-source licentie: MIT

- Zelf bijdragen?
  - Issues voor vragen
  - Documentatie
  - Code

## Quick start

In [ ]:
from pydov.search.boring import BoringSearch

from pydov.util.location import Within, Box

from owslib.fes2 import PropertyIsGreaterThan

boring_search = BoringSearch()

dataframe = boring_search.search(
    query=PropertyIsGreaterThan(propertyname='diepte_tot_m', literal='550'),
    location=Within(Box(107500, 202000, 108500, 203000, epsg=31370))
)

dataframe

## Datasets selecteren
> Meer info: https://pydov.readthedocs.io/en/stable/select_datasets.html

Om data op te halen moet je eerst een keuze maken welke datasets je wil bevragen.

### Zoekobjecten

Elk van de beschikbare datasets heeft een bijhorend zoekobject waarmee je de data kan bevragen.

Volgende code maakt drie zoekobjecten aan, om respectievelijk data over Boringen, Monsters en Observaties te kunnen bevragen.

In [ ]:
from pydov.search.boring import BoringSearch
from pydov.search.monster import MonsterSearch
from pydov.search.observatie import ObservatieSearch

boring_search = BoringSearch()
monster_search = MonsterSearch()
observatie_search = ObservatieSearch()

_Oefening: met welk zoekobject kan je grondwater peilmetingen terugvinden?_
<details>
<summary>Antwoord</summary>
from pydov.search.grondwaterfilter import GrondwaterFilterSearch

grondwaterfilter_search = GrondwaterFilterSearch()
</details>

### Object types
> Meer info: https://pydov.readthedocs.io/en/stable/output_fields.html#customizing-object-types-and-subtypes

Elk zoekobject is gekoppeld aan een object type, dat bepaalt welke velden er standaard in het resultaat beschikbaar zijn.

Bij het aanmaken van het zoekobject kan je optioneel het objecttype opgeven, om zo te kunnen beïnvloeden welke velden er teruggegeven zullen worden. Volgende code is equivalent aan de eerdere versie van `observatie_search`:

In [ ]:
from pydov.search.observatie import ObservatieSearch

from pydov.types.observatie import Observatie

observatie_search = ObservatieSearch(
    objecttype=Observatie
)

Volgend schema verduidelijkt het verschil tussen zoekobjecten, objecttypes en subtypes. Een zoekobject bepaalt op welke velden er gezocht kan worden, een objecttype bepaalt welke velden er in het resultaat teruggegeven kunnen worden. Per record uit het zoekresultaat is er maximum één resultaat uit het (hoofd) objecttype, en kunnen er meerdere records uit het subtype zijn.

![objecttypes](../objecttypes.svg)

### Fieldsets

Bij sommige objecttypes zijn er extra velden beschikbaar die niet standaard aanwezig zijn in het resultaat, maar die eenvoudig toegevoegd kunnen worden. Via de methode `get_fieldsets()` bij een objecttype kan je opvragen welke sets beschikbaar zijn.

In [ ]:
from pydov.search.observatie import ObservatieSearch
from pydov.types.observatie import Observatie

Observatie.get_fieldsets()

In [ ]:
from pydov.types.observatie import ObservatieDetails

observatie_search = ObservatieSearch(
    objecttype=Observatie.with_extra_fields(ObservatieDetails)
)

_Oefening: welke fieldsets zijn er beschikbaar voor Boringen?_
<details>
<summary>Antwoord</summary>

__MethodeXyz__

from pydov.types.boring import Boring

Boring.get_fieldsets()
</details>

### Subtypes

Bij sommige objecttypes zijn er extra subtypes beschikbaar, die gebruikt kunnen worden in plaats van het standaard subtype. Via de methode `get_subtypes()` bij een objecttype kan je opvragen welke subtypes beschikbaar zijn.

In [ ]:
from pydov.search.observatie import ObservatieSearch
from pydov.types.observatie import Observatie

Observatie.get_subtypes()

In [ ]:
from pydov.types.observatie import SecundaireParameter

observatie_search = ObservatieSearch(
    objecttype=Observatie.with_subtype(SecundaireParameter)
)

_Oefening: welke subtypes zijn er beschikbaar voor GrondwaterFilters?_
<details>
<summary>Antwoord</summary>

__Gxg en Peilmeting__

from pydov.types.grondwaterfilter import GrondwaterFilter

GrondwaterFilter.get_subtypes()
</details>

## Zoeken op locatie
> Meer info: https://pydov.readthedocs.io/en/stable/query_location.html

Geografisch zoeken kan met de `location` parameter van de `search` methode. Je geeft hieraan een geografische filter en een geometrie-object mee, ofwel een filter factory op basis van een geodataframe of vector GIS bestand.

### Overlap met rechthoek

Zoeken naar objecten die overlappen met een rechthoekig gebied is eenvoudig:


In [ ]:
from pydov.search.observatie import ObservatieSearch
from pydov.util.location import Within, Box

observatie_search = ObservatieSearch()

observatie_search.search(
    location=Within(Box(minx=200000, miny=211000, maxx=201000, maxy=212000, epsg=31370))
)

_Oefening: hoeveel monsters bevinden zich in de rechthoek met coördinaten [150000, 150000, 175000, 175000]?_
<details>
<summary>Antwoord</summary>

__8669__

from pydov.search.monster import MonsterSearch
from pydov.util.location import Within, Box

monster_search = MonsterSearch()

monster_search.search(
    location=Within(Box(minx=150000, miny=150000, maxx=175000, maxy=175000, epsg=31370))
)
</details>

### Buffer rond puntlocatie

Je kan ook zoeken op een cirkelvormige buffer rondom een puntlocatie:

In [ ]:
from pydov.search.observatie import ObservatieSearch
from pydov.util.location import WithinDistance, Point

observatie_search = ObservatieSearch()

observatie_search.search(
    location=WithinDistance(
        Point(x=200000, y=205000, epsg=31370),
        distance=500)
)

_Oefening: hoeveel observaties bevinden zich binnen een straal van 250 meter rond het belfort van Gent?_
<details>
<summary>Antwoord</summary>

__7__

from pydov.search.observatie import ObservatieSearch
from pydov.util.location import WithinDistance, Point

observatie_search = ObservatieSearch()

observatie_search.search(
    location=WithinDistance(
        Point(x=3.725278, y=51.053889, epsg=4326),
        distance=250)
)
</details>

### GeoPandas geodataframe

Je kan ook geografisch zoeken op basis van een Geodataframe. Dit kan je gebruiken in een GeopandasFilter factory, tesamen met een locatiefilter.

Hieronder maken we eerst een geodataframe aan:

In [8]:
import geopandas as gpd

shapefile = "../../tests/data/util/location/polygon_multiple_31370.shp"

geodataframe = gpd.read_file(shapefile)
geodataframe["name"] = ["site 1", "site 2"]
geodataframe

,gml_id,geometry,name
0,polygon_multiple_31370.0,"POLYGON ((108636.15 194960.844, 109195.574 195...",site 1
1,polygon_multiple_31370.1,"POLYGON ((107485.786 196741.544, 108297.344 19...",site 2


Dit kunnen we nu gebruiken in een pydov zoekopdracht, bijvoorbeeld om boringen te vinden:

In [10]:
from pydov.search.boring import BoringSearch
from pydov.util.location import Within, GeopandasFilter

boring_search = BoringSearch()

boring_search.search(
    location=GeopandasFilter(geodataframe, Within)
)

[000/001] .
[000/018] ccccccccccccccccc.


,pkey_boring,boornummer,x,y,mv_mtaw,start_boring_mtaw,gemeente,diepte_boring_van,diepte_boring_tot,datum_aanvang,uitvoerder,boorgatmeting,diepte_methode_van,diepte_methode_tot,boormethode
0,https://www.dov.vlaanderen.be/data/boring/2018...,B/4-104356,108025.00,196593.00,8.05,8.05,Gent,0.0,7.0,NaN,NaN,False,0.0,0.0,onbekend
1,https://www.dov.vlaanderen.be/data/boring/2019...,1718-B-180092,107947.29,196640.52,7.96,7.96,NaN,0.0,8.0,2019-03-11,Verhofste,False,0.0,8.0,spoelboring
2,https://www.dov.vlaanderen.be/data/boring/2020...,1718-B-190135,107991.00,196706.00,8.40,8.40,NaN,0.0,8.0,2020-05-15,Verhofste,False,0.0,8.0,spoelboring
3,https://www.dov.vlaanderen.be/data/boring/2022...,1407-B0863,107842.53,196371.08,7.38,7.38,Gent,0.0,4.0,2022-06-08,De Backer Putboringen,False,0.0,4.0,spoelboring
4,https://www.dov.vlaanderen.be/data/boring/2023...,1718-B220073,108144.95,196771.09,9.18,9.18,NaN,0.0,4.0,2023-04-13,Verhofste,False,0.0,4.0,spoelboring
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,https://www.dov.vlaanderen.be/data/boring/1893...,kb22d55e-B44,108900.00,194425.00,6.00,6.00,Destelbergen,0.0,21.0,1893-01-01,Behiels-(Lemmens)-Wetteren,False,0.0,21.0,onbekend
64,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B102,107618.00,196709.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
65,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B103,107791.00,196516.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
66,https://www.dov.vlaanderen.be/data/boring/1895...,kb22d55e-B400,109050.00,194990.00,7.00,7.00,Destelbergen,0.0,0.0,1895-01-01,onbekend,False,0.0,0.0,onbekend


We kunnen ook rechtstreeks een GIS bestand (bijvoorbeeld Shapefile) gebruiken met een GeometryFilter factory:

In [11]:
from pydov.search.boring import BoringSearch
from pydov.util.location import Within, GeometryFilter

boring_search = BoringSearch()

studiegebied = '../../tests/data/util/location/polygon_multiple_31370.shp'

boring_search.search(
    location=GeometryFilter(studiegebied, Within)
)

[000/001] .
[000/018] cccccccccccccccccc


,pkey_boring,boornummer,x,y,mv_mtaw,start_boring_mtaw,gemeente,diepte_boring_van,diepte_boring_tot,datum_aanvang,uitvoerder,boorgatmeting,diepte_methode_van,diepte_methode_tot,boormethode
0,https://www.dov.vlaanderen.be/data/boring/2018...,B/4-104356,108025.00,196593.00,8.05,8.05,Gent,0.0,7.0,NaN,NaN,False,0.0,0.0,onbekend
1,https://www.dov.vlaanderen.be/data/boring/2019...,1718-B-180092,107947.29,196640.52,7.96,7.96,NaN,0.0,8.0,2019-03-11,Verhofste,False,0.0,8.0,spoelboring
2,https://www.dov.vlaanderen.be/data/boring/2020...,1718-B-190135,107991.00,196706.00,8.40,8.40,NaN,0.0,8.0,2020-05-15,Verhofste,False,0.0,8.0,spoelboring
3,https://www.dov.vlaanderen.be/data/boring/2022...,1407-B0863,107842.53,196371.08,7.38,7.38,Gent,0.0,4.0,2022-06-08,De Backer Putboringen,False,0.0,4.0,spoelboring
4,https://www.dov.vlaanderen.be/data/boring/2023...,1718-B220073,108144.95,196771.09,9.18,9.18,NaN,0.0,4.0,2023-04-13,Verhofste,False,0.0,4.0,spoelboring
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,https://www.dov.vlaanderen.be/data/boring/1893...,kb22d55e-B44,108900.00,194425.00,6.00,6.00,Destelbergen,0.0,21.0,1893-01-01,Behiels-(Lemmens)-Wetteren,False,0.0,21.0,onbekend
64,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B102,107618.00,196709.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
65,https://www.dov.vlaanderen.be/data/boring/1894...,kb22d55e-B103,107791.00,196516.00,7.50,7.50,Gent,0.0,0.0,1894-01-01,onbekend,False,0.0,0.0,onbekend
66,https://www.dov.vlaanderen.be/data/boring/1895...,kb22d55e-B400,109050.00,194990.00,7.00,7.00,Destelbergen,0.0,0.0,1895-01-01,onbekend,False,0.0,0.0,onbekend


_Oefening: hoeveel observaties bevinden zich binnenin of binnen een straal van 200 meter rondom het studiegebied?_
<details>
<summary>Antwoord</summary>

__303__

from pydov.search.observatie import ObservatieSearch
from pydov.util.location import WithinDistance, GeometryFilter

observatie_search = ObservatieSearch()

observatie_search.search(
    location=GeometryFilter(studiegebied, WithinDistance, {'distance': 200})
)
</details>

## Zoeken op attributen
> Meer info: https://pydov.readthedocs.io/en/stable/query_attribute.html

Naast zoeken op locatie, kan je ook zoeken naar objecten met bepaalde eigenschappen.

### Beschikbare zoekvelden

Om na te gaan welke attributen (velden) er beschikbaar zijn in een datatype, kan je de methode `get_fields()` gebruiken. Om specifiek de velden op te vragen waarop je kan zoeken gebruik je de volgende code:

In [ ]:
from pydov.search.monster import MonsterSearch

monster_search = MonsterSearch()

monster_search.get_fields(query=True)

### Attribuut gelijk aan

Zoeken op attribuutwaarden kan met de zoekoperatoren uit `owslib.fes2`: PropertyIsEqualTo, PropertyIsNotEqualTo, PropertyIsNull, PropertyIsLike, PropertyIsLessThan, PropertyIsLessThanOrEqualTo, PropertyIsGreaterThan, PropertyIsGreaterThanOrEqualTo, PropertyIsBetween.

Om bijvoorbeeld monsters op te vragen waarbij het monstertype ongeroerd is kan je volgende code gebruiken. De parameter `max_features` zorgt ervoor dat er maximaal dit aantal features teruggegeven worden, dat is handig om je query te testen op een subset van de data.

In [ ]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import PropertyIsEqualTo

monster_search = MonsterSearch()

monster_search.search(
    query=PropertyIsEqualTo('monstertype', 'ongeroerd'),
    max_features=10
)

### Attribuut groter dan of gelijk aan

Om bijvoorbeeld alle monsters genomen sinds 1/1/2025 op te vragen gebruik je volgende code:

In [ ]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import PropertyIsGreaterThanOrEqualTo

monster_search = MonsterSearch()

monster_search.search(
    query=PropertyIsGreaterThanOrEqualTo(
        'datum_monstername', '2025-01-01'),
    max_features=10
)

### Attribuutfilters combineren

Het is ook mogelijk om verschillende filters (ook genest) te combineren met de And, Or, Not operatoren uit `owslib.fes2`.

Om de twee voorgaande zoekopdrachten te combineren gebruik je bijvoorbeeld:

In [ ]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import (
    And, PropertyIsEqualTo, PropertyIsGreaterThanOrEqualTo)

monster_search = MonsterSearch()

monster_search.search(
    query=And([
        PropertyIsGreaterThanOrEqualTo('datum_monstername', '2025-01-01'),
        PropertyIsEqualTo('monstertype', 'ongeroerd')
    ])
)

_Oefening: hoeveel observaties werden geobserveerd in het VELD vanaf 1/1/2025 op een diepte van minstens 175 meter?_
<details>
<summary>Antwoord</summary>

__8__

from pydov.search.observatie import ObservatieSearch

from owslib.fes2 import (
    And, PropertyIsEqualTo, PropertyIsGreaterThanOrEqualTo)

observatie_search = ObservatieSearch()

observatie_search.search(
    query=And([
        PropertyIsGreaterThanOrEqualTo('fenomeentijd', '2025-01-01'),
        PropertyIsGreaterThanOrEqualTo('diepte_van_m', '175'),
        PropertyIsEqualTo('herkomst', 'VELD')
    ])
)
</details>

### Attribuutfilters combineren met locatie

Tenslotte kan je attribuutfilters ook combineren met een locatiefilter. Hierbij krijg je enkel resultaten die aan beide filters voldoen.

In [ ]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import (
    And, PropertyIsEqualTo, PropertyIsGreaterThanOrEqualTo)

monster_search = MonsterSearch()

monster_search.search(
    query=And([
        PropertyIsGreaterThanOrEqualTo('datum_monstername', '2025-01-01'),
        PropertyIsEqualTo('monstertype', 'ongeroerd')
    ]),
    location=Within(Box(18000, 200000, 220000, 230000, epsg=31370))
)

## Resultaatvelden selecteren

Als je minder (of: meer) velden nodig hebt dan er standaard aanwezig zijn in het resultaat, kan je de parameter `return_fields` gebruiken om deze velden aan te passen.

Je vraagt best enkel de velden op die je nodig hebt, dit zorgt ervoor dat de services niet onnodig belast worden en dat het resultaat ook sneller beschikbaar zal zijn.

In [ ]:
from pydov.search.monster import MonsterSearch

monster_search = MonsterSearch()

monster_search.get_fields()

df_monsters = monster_search.search(
    max_features=10,
    return_fields=['pkey_monster', 'naam', 'diepte_van_m', 'diepte_tot_m']
)

df_monsters

_Oefening: hoe diep is de diepste boring in Poperinge?_
<details>
<summary>Antwoord</summary>

__665 meter__

from pydov.search.boring import BoringSearch

boring_search = BoringSearch()

df_boringen = boring_search.search(
    query=PropertyIsEqualTo('gemeente', 'Poperinge'),
    return_fields=['diepte_boring_tot']
)

float(df_boringen.diepte_boring_tot.max())
</details>

_Extra: hoe diep is de diepste boring in de DOV databank?_
<details>
<summary>Antwoord</summary>

__4905 meter__

from pydov.search.boring import BoringSearch
from owslib.fes2 import SortBy, SortProperty

bs = BoringSearch()
bs.search(sort_by=SortBy([SortProperty('diepte_boring_tot', 'DESC')]),
          max_features=1,
          return_fields=('pkey_boring', 'datum_aanvang', 'uitvoerder', 'gemeente', 'diepte_boring_tot'))
</details>

### Geometrie toevoegen

Standaard bevatten de resultaat dataframes enkel attribuutwaarden. Om ook de geometrie op te vragen kan je dit veld toevoegen aan de lijst met `return_fields`.

Om de naam van de geometrie kolom te bekomen kan je volgende opdracht gebruiken:

In [ ]:
from pydov.search.monster import MonsterSearch

monster_search = MonsterSearch()

monster_search.get_fields(type='geometry')

Vervolgens kan je deze toevoegen als `GeometryReturnField`, waarbij je naast de naam ook het gewenst coördinatensysteem opgeeft:

In [ ]:
from pydov.search.fields import GeometryReturnField

df_monsters = monster_search.search(
    max_features=10,
    return_fields=['pkey_monster', 'naam', GeometryReturnField('geom', epsg=31370)]
)

df_monsters

Dit resultaat kan je eenvoudig omzetten naar een geodataframe:

In [ ]:
gdf_monsters = gpd.GeoDataFrame(df_monsters, geometry='geom', crs='EPSG:31370')
gdf_monsters.explore()

## Datasets combineren

Naast de uitgebreide zoekmogelijkheden zit de kracht van pydov in de mogelijkheden om verschillende datasets te combineren. Zo kan je de resultaten van één dataset gebruiken om verder te zoeken en zo verschillende datasets linken aan elkaar.

Zo kan je bijvoorbeeld op zoek gaan naar recente ongeroerde monsters van materiaalklasse 'sediment' in je studiegebied:

In [ ]:
from pydov.search.monster import MonsterSearch

from owslib.fes2 import (
    And, PropertyIsEqualTo, PropertyIsGreaterThanOrEqualTo)

monster_search = MonsterSearch()

df_monsters = monster_search.search(
    query=And([
        PropertyIsGreaterThanOrEqualTo('datum_monstername', '2025-01-01'),
        PropertyIsEqualTo('monstertype', 'ongeroerd'),
        PropertyIsEqualTo('materiaalklasse', 'sediment')
    ]),
    location=Within(Box(18000, 200000, 220000, 230000, epsg=31370)),
)

df_monsters

En vervolgens de boringen opvragen waarvan deze monsters genomen zijn:

In [ ]:
from pydov.search.boring import BoringSearch

from pydov.util.query import Join

boring_search = BoringSearch()

df_boringen = boring_search.search(
    query=Join(df_monsters, on='pkey_boring', using='pkey_parents'),
    return_fields=['pkey_boring', 'boornummer', 'x', 'y', 'start_boring_mtaw', 'diepte_boring_van', 'diepte_boring_tot']
)

df_boringen